In [ ]:

import numpy as np

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans


# ==========================================
# STEP 1: GENERATE DATA
# ==========================================

def make_data(seed=0):

    X, y = make_blobs(
        n_samples=200,
        centers=3,
        n_features=2,
        cluster_std=0.8,
        random_state=seed
    )

    X = X.astype(float)

    # Stretch second feature
    X[:, 1] = X[:, 1] * 50.0

    return X, y


# ==========================================
# STEP 2: Z-SCORE STANDARDIZATION
# ==========================================

def zscore_standardize(X):

    X = np.asarray(X, dtype=float)

    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)

    # Handle constant columns
    std[std == 0] = 1.0

    return (X - mean) / std


# ==========================================
# STEP 3: CLUSTER INERTIA
# ==========================================

def cluster_inertia(X, k, seed=0):

    model = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=seed
    )

    model.fit(X)

    return float(model.inertia_)


# ==========================================
# STEP 4: ELBOW METHOD
# ==========================================

def elbow_inertias(X, k_range, seed=0):

    inertias = []

    for k in k_range:

        inertia = cluster_inertia(
            X,
            k,
            seed
        )

        inertias.append(inertia)

    return inertias


# ==========================================
# STEP 5: SILHOUETTE METRIC FROM SCRATCH
# ==========================================

def silhouette_metric(X, labels):

    X = np.asarray(X, dtype=float)
    labels = np.asarray(labels)

    n = len(X)

    # Calculate pairwise Euclidean distances
    distances = np.sqrt(
        np.sum(
            (X[:, np.newaxis, :] - X[np.newaxis, :, :]) ** 2,
            axis=2
        )
    )

    unique_labels = np.unique(labels)

    scores = []

    for i in range(n):

        current_label = labels[i]

        # Points belonging to the same cluster
        same_cluster = np.where(
            labels == current_label
        )[0]

        # Exclude the current point itself
        same_cluster = same_cluster[
            same_cluster != i
        ]

        # Singleton cluster: silhouette is zero
        if len(same_cluster) == 0:
            scores.append(0.0)
            continue

        # Mean intra-cluster distance
        a = np.mean(
            distances[i, same_cluster]
        )

        # Mean distance to each other cluster
        other_distances = []

        for label in unique_labels:

            if label == current_label:
                continue

            other_cluster = np.where(
                labels == label
            )[0]

            mean_distance = np.mean(
                distances[i, other_cluster]
            )

            other_distances.append(mean_distance)

        # Minimum mean distance to another cluster
        b = min(other_distances)

        denominator = max(a, b)

        if denominator == 0:
            score = 0.0
        else:
            score = (b - a) / denominator

        scores.append(score)

    return float(np.mean(scores))


# ==========================================
# STEP 6: FIND BEST K USING SILHOUETTE
# ==========================================

def best_k_by_silhouette(X, k_range, seed=0):

    best_k = None
    best_score = -np.inf

    for k in k_range:

        model = KMeans(
            n_clusters=k,
            n_init=10,
            random_state=seed
        )

        labels = model.fit_predict(X)

        score = silhouette_metric(
            X,
            labels
        )

        if score > best_score:
            best_score = score
            best_k = k

    return int(best_k)


# ==========================================
# STEP 7: GENERATE DATA
# ==========================================

X, y = make_data(0)

print("Original Dataset Shape:")
print("X:", X.shape)
print("y:", y.shape)


# ==========================================
# STEP 8: STANDARDIZE DATA
# ==========================================

Z = zscore_standardize(X)

print("\nStandardized Mean:")
print(Z.mean(axis=0).round(6))

print("\nStandardized Standard Deviation:")
print(Z.std(axis=0).round(6))


# ==========================================
# STEP 9: TEST CONSTANT COLUMN
# ==========================================

test_data = np.array([
    [1, 10, 5],
    [2, 20, 5],
    [3, 30, 5],
    [4, 40, 5]
], dtype=float)

test_scaled = zscore_standardize(test_data)

print("\nConstant Column Test:")
print(test_scaled)

print("Contains NaN:", np.isnan(test_scaled).any())
print("Contains Inf:", np.isinf(test_scaled).any())


# ==========================================
# STEP 10: ELBOW METHOD
# ==========================================

k_range = [2, 3, 4, 5, 6]

inertias = elbow_inertias(
    Z,
    k_range,
    seed=0
)

print("\nElbow Inertias:")
print([round(v, 1) for v in inertias])


# ==========================================
# STEP 11: SILHOUETTE SCORE
# ==========================================

km = KMeans(
    n_clusters=3,
    n_init=10,
    random_state=0
)

labels = km.fit_predict(Z)

sil_score = silhouette_metric(
    Z,
    labels
)

print("\nSilhouette Score:")
print(round(sil_score, 3))


# ==========================================
# STEP 12: FIND OPTIMAL K
# ==========================================

best_k = best_k_by_silhouette(
    Z,
    k_range,
    seed=0
)

print("\nBest K:")
print(best_k)


# ==========================================
# STEP 13: COMPARE SCALING EFFECT
# ==========================================

km_raw = KMeans(
    n_clusters=3,
    n_init=10,
    random_state=0
)

raw_labels = km_raw.fit_predict(X)

raw_silhouette = silhouette_metric(
    X,
    raw_labels
)

km_scaled = KMeans(
    n_clusters=3,
    n_init=10,
    random_state=0
)

scaled_labels = km_scaled.fit_predict(Z)

scaled_silhouette = silhouette_metric(
    Z,
    scaled_labels
)

print("\nRaw Data Silhouette:")
print(round(raw_silhouette, 3))

print("\nScaled Data Silhouette:")
print(round(scaled_silhouette, 3))

print(
    "\nScaling Improves Silhouette:",
    scaled_silhouette > raw_silhouette
)


# ==========================================
# STEP 14: FINAL VERIFICATION
# ==========================================

print("\nFinal Verification:")

print("Correct data shape:", X.shape == (200, 2))

print(
    "Mean approximately zero:",
    np.allclose(Z.mean(axis=0), 0, atol=1e-6)
)

print(
    "Std approximately one:",
    np.allclose(Z.std(axis=0), 1, atol=1e-6)
)

print(
    "Elbow non-increasing:",
    all(
        inertias[i] >= inertias[i + 1] - 1e-8
        for i in range(len(inertias) - 1)
    )
)

print(
    "Silhouette in valid range:",
    -1 <= sil_score <= 1
)

print("Best K is 3:", best_k == 3)
